In [18]:
import sys
sys.path.append("..")
from src.ingest.itf import parse_itf, to_numeric, parse_itf_to_parts

In [19]:
parsed_sample, stats = parse_itf("../data/itf_slice100k.txt")
print(len(parsed_sample), "rows parsed")
print(stats)

98300 rows parsed
{'parsed': 98300, 'skipped': 1410}


In [20]:
print(parsed_sample.columns.tolist())

['desig', 'date_str', 'ra_str', 'dec_str', 'fmt', 'obscode']


In [21]:
%%time
#this cell is for testing the speed of my string --> numerical conversion functions
converted = to_numeric(parsed_sample) #creates a new variable that stores the modified rows

CPU times: user 18.5 s, sys: 341 ms, total: 18.9 s
Wall time: 20.7 s


In [22]:
print(parsed_sample.memory_usage(deep=True).sum() / 1e6, "MB for 100k rows")

10.335501 MB for 100k rows


In [23]:
print(parsed_sample.columns.tolist())

['desig', 'date_str', 'ra_str', 'dec_str', 'fmt', 'obscode']


In [24]:
%%time
KEEP = ["desig", "ra_str", "dec_str", "date_str", "obscode", "fmt"]
1

CPU times: user 23 μs, sys: 2 μs, total: 25 μs
Wall time: 30 μs


1

In [25]:
%%time
#this cell should parse the ITF, save it to an itf_numeric file (which was deleted, but is something I should later comment out)
#itf_parsed = parse_itf_to_parts("../data/itf.txt", "../data/itf_numeric", 100_000, keep=KEEP)

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 8.82 μs


In [26]:
#%%time
#this cell will convert the parqet files generated from the full parsing of the ITF
import os, glob
import pandas as pd

src_dir = "../data/itf_numeric" #the text parts I already have
out_dir = "../data/itf_converted" #where the converted parts will go
#os.makedirs(out_dir, exist_ok=True)

KEEP = ["desig", "ra_deg", "dec_deg", "time", "obscode"] #what I want to keep once the new columns have been created

parts = sorted(glob.glob(f"{src_dir}/part_*.parquet"))
#print(len(parts), "parts to convert")

#for i, p in enumerate(parts):
#    df = to_numeric(pd.read_parquet(p))
#    df[KEEP].to_parquet(f"{out_dir}/part_{i:03d}.parquet")
#    if i % 10 == 0:
#        print(f"part {i:03d} done")
#    del df

#print("all parts converted")


In [17]:
%%time
#this cell will stitch all of the new files in itf_converted together.
converted = sorted(glob.glob(f"{out_dir}/part_*.parquet"))

full = pd.concat([pd.read_parquet(p) for p in converted], ignore_index=True)

print(f"{len(full):,} rows")
full.to_parquet("../data/itf_numeric_full.parquet")

9,054,785 rows
CPU times: user 2.73 s, sys: 1.19 s, total: 3.92 s
Wall time: 19.3 s
